# TAC Desk: Predict ticket resolution time

At triage, can we estimate how many hours a ticket will take to resolve? This is **supervised learning** because historical tickets come with known answers, and **regression** because the answer is a number of hours.

Use the project's Python environment and run the cells in order from the workspace root or the `notebooks` folder.

## 1. Load and inspect the CSV

`head()` previews five rows, `columns` lists the headers, and `info()` shows column types and non-null counts.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

csv_path = workspace_root / "data" / "tac_tickets.csv"
tickets = pd.read_csv(csv_path)

display(tickets.head())
print("Column names:", tickets.columns.tolist())
tickets.info()

## 2. Choose the inputs and target

- **Features (`X`):** number of devices affected and the software version's age in months.
- **Target (`y`):** actual resolution time in hours. We know it for past tickets, but not for a new one at triage.

We leave outcomes such as `resolution_code` and `sla_breached` out of the inputs because they aren't known at triage. `describe()` summarizes only the three columns we'll use.

In [ ]:
feature_columns = ["devices_affected", "software_version_months"]
target_column = "resolution_hours"

X = tickets[feature_columns]
y = tickets[target_column]

display(tickets[feature_columns + [target_column]].describe().round(2))

## 3. Split the data and train the models

Train on 80% of the tickets and hold back 20% for testing. A fixed random seed gives everyone the same split. Linear regression learns from the two features; the baseline ignores them and always predicts the average resolution time from the training set.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training tickets: {len(X_train):,} | Test tickets: {len(X_test):,}")

model = LinearRegression()
model.fit(X_train, y_train)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train);

## 4. Evaluate on the test set

Both models predict for the same held-out tickets. **Mean absolute error (MAE)** tells us how many hours off the predictions were on average, whether too high or too low. Lower is better. Did using the two features beat the baseline?

In [ ]:
from sklearn.metrics import mean_absolute_error

model_predictions = model.predict(X_test)
baseline_predictions = baseline.predict(X_test)

model_mae = mean_absolute_error(y_test, model_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Linear regression MAE: {model_mae:.2f} hours")
print(f"Training-mean baseline MAE: {baseline_mae:.2f} hours")

## 5. Predict resolution time for a new ticket

Give the trained model one row with the same two feature names. Change either input and rerun this cell to see how the predicted resolution time changes.

In [ ]:
new_ticket = pd.DataFrame({
    "devices_affected": [5],
    "software_version_months": [18],
})

predicted_hours = model.predict(new_ticket)[0]
print(f"Predicted resolution time: {predicted_hours:.2f} hours")